# Day 04 — File I/O · JSON · String Operations
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

> **What you'll learn:** `with` statement · reading and writing text files · `json.loads()` · `json.dumps()` · safe dict access with `.get()` · reading CSV datasets · string cleaning methods

> **Why this matters:** System prompts live in `.txt` files — not hardcoded in code. LLM responses come back as JSON strings you must parse. String methods clean raw database text before it reaches the LLM.

> **Connection to Day 03:** The functions you built yesterday (`build_system_prompt`, `build_api_request`) are now loaded from files and filled with real data from CSV and JSON.


---


## 1 — Imports and Path Setup

### Why `__file__` fails in Jupyter

`__file__` is set automatically when Python **executes a `.py` file**.  
In a Jupyter notebook there is no file — code runs in a live kernel — so `__file__` is not defined.

| Context | `__file__` | Solution |
|---------|-----------|----------|
| Running `python day04.py` | Defined ✅ | `Path(__file__).parent` |
| Jupyter notebook cell | **NameError** ❌ | `Path.cwd()` |

Use the `try/except` pattern below — works in both contexts automatically.


In [ ]:
import json           # parse and produce JSON strings — LLM responses arrive as JSON
import os             # file path operations (also Path covers most use cases)
import csv            # read CSV datasets — customers, products, reviews
from pathlib import Path   # Path('data') / 'file.txt' is cleaner than os.path.join()
from typing import Any     # type hint for values that can be any type (used in safe_get_field)

# ── Portable path setup ──────────────────────────────────────
# __file__ = absolute path of the current .py file (only exists in .py scripts)
# In Jupyter: __file__ is not defined → NameError → fall back to Path.cwd()
try:
    BASE_DIR = Path(__file__).parent.parent   # .py script: go up to project root
except NameError:
    BASE_DIR = Path.cwd()                      # Jupyter: use current working directory

# / operator on Path objects joins paths — cleaner than os.path.join()
DATA_DIR   = BASE_DIR / "data" / "datasets"  # where CSVs live
PROMPT_DIR = BASE_DIR / "prompts"            # where .txt and .json prompt files live

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"PROMPT_DIR : {PROMPT_DIR}")


---


## 2 — The `with` Statement — Safe Resource Management

The `with` statement guarantees a resource (file, DB connection, HTTP session) is properly **closed even if an exception occurs** inside the block.

```python
# WITHOUT with — dangerous:
f = open("file.txt")
data = f.read()      # if this raises, f.close() is NEVER called — file stays locked
f.close()

# WITH the with statement — safe:
with open("file.txt") as f:
    data = f.read()  # if this raises, Python still calls f.close() automatically
```

| File mode | String | Use for |
|-----------|--------|---------|
| Read | `"r"` | Reading an existing file (default) |
| Write | `"w"` | Writing — **overwrites** if file exists |
| Append | `"a"` | Adding to end — creates file if missing |

> Always specify `encoding="utf-8"` to avoid platform differences between Mac, Windows, Linux.

> **Day 09 preview:** `async with` is the same concept for async resources — same syntax, non-blocking.


### `load_system_prompt()` — read prompt from `.txt` file


In [ ]:
def load_system_prompt(filename: str = "system_prompt.txt") -> str:
    """
    Load a system prompt from a .txt file in the prompts/ directory.

    Keeping prompts in files (not hardcoded in code) means:
    - Non-technical team members can edit prompts without touching Python
    - Prompts are version-controlled independently from business logic
    - Different prompts can be swapped based on environment (dev/prod)
    """
    # PROMPT_DIR / filename  →  e.g. /project/prompts/system_prompt.txt
    # Path / operator builds the full path safely on any OS
    filepath = PROMPT_DIR / filename

    # with open(...) as f:  →  the context manager
    # "r"            = read mode (default — included here for clarity)
    # encoding='utf-8' = always specify — avoids silent corruption on Windows
    #                    ('cp1252' is the Windows default and breaks emoji/special chars)
    # The file is guaranteed to be closed when the with block exits,
    # even if f.read() raises an exception
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()   # f.read() loads the ENTIRE file into one string

    # .strip() removes leading and trailing whitespace characters:
    # spaces, \n (newline), \t (tab), \r (carriage return)
    # Text editors almost always add a trailing newline — .strip() removes it
    # so the prompt starts and ends cleanly when injected into the API request
    return content.strip()


In [ ]:
# ── Demo: load_system_prompt() ─────────────────────────────
import tempfile

# Write a sample prompt to a temp file (simulates prompts/system_prompt.txt)
sample_prompt = """## Role
You are a customer support assistant for ShopSmart e-commerce.

## Task
Answer using ONLY the information provided. Never invent details."""

# mode='w' creates the file | suffix='.txt' | delete=False keeps it after closing
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as tmp:
    tmp.write(sample_prompt)
    tmp_path = tmp.name

# Now read it back — exactly what load_system_prompt() does at startup
with open(tmp_path, 'r', encoding='utf-8') as f:
    loaded = f.read().strip()   # .strip() removes trailing newline text editors add

print(f"Loaded {len(loaded)} characters:")
print(loaded)


### `save_llm_response_log()` — append call record to JSONL log


In [ ]:
def save_llm_response_log(
    query: str,
    response: str,
    log_file: str = "llm_calls.log",
) -> None:
    """
    Append one LLM call record to a JSONL log file.
    JSONL = JSON Lines — one JSON object per line, machine-readable.
    """
    log_path = BASE_DIR / log_file

    # "a" = append mode:
    #   - if file exists  → adds to the END without overwriting previous entries
    #   - if file missing → creates it automatically
    # This is why logs survive across multiple program runs
    with open(log_path, "a", encoding="utf-8") as f:

        # json.dumps() converts a Python dict → JSON string
        # direction: dict  →  string  (opposite of json.loads)
        # Result: '{"query": "Where is my order?", "response": "In Transit."}'
        log_entry = json.dumps({
            "query"   : query,
            "response": response,
        })

        # Write the JSON string + newline
        # Each call appends exactly one line → the file is valid JSONL format
        # Later you can parse it with: for line in f: json.loads(line)
        f.write(log_entry + "\n")


In [ ]:
# ── Demo: save_llm_response_log() ──────────────────────────
import tempfile, os

log_path = tempfile.mktemp(suffix='.log')   # temp path — file not created yet

# Append two log entries
# TRACE:
# Call 1 → 'a' mode: file missing → creates it
#           writes: {"query": "Where is my order?", "response": "In Transit."}\n
# Call 2 → 'a' mode: file exists → appends to END (does not overwrite line 1)
#           writes: {"query": "Can I return?", "response": "30 days."}\n
for query, response in [
    ('Where is my order?',  'Your order ORD-3042 is In Transit.'),
    ('Can I return this?',  'Returns accepted within 30 days.'),
]:
    with open(log_path, 'a', encoding='utf-8') as f:
        entry = json.dumps({'query': query, 'response': response})
        f.write(entry + '\n')   # each call = exactly one line (JSONL)

# Read back every line — each is a JSON string we parse back to a dict
# TRACE:
# line 0 → '{"query": "Where is my order?", ...}\n'
#           .strip() removes \n → json.loads() converts string → dict
# line 1 → second entry
with open(log_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        record = json.loads(line.strip())
        print(f"[{i+1}] Q: {record['query']}")
        print(f"     A: {record['response']}")

os.unlink(log_path)   # clean up temp file


---


## 3 — JSON Parsing — `json.loads()` and `json.dumps()`

| Function | Direction | Use |
|----------|-----------|-----|
| `json.loads(string)` | string → Python dict | Parse LLM response |
| `json.dumps(dict)` | Python dict → string | Write log entries, build API payloads |
| `json.load(file)` | file object → Python dict | Load config/examples from `.json` file |
| `json.dump(dict, file)` | Python dict → file | Save results to `.json` file |

### Why LLM responses need parsing

The OpenAI API returns a **string** — even when the LLM was instructed to produce JSON.  
You must call `json.loads()` to convert it into a Python dict before you can access any fields.

```python
raw = '{"category": "URGENT", "confidence": "high"}'   # this is a str
raw["category"]                 # TypeError — strings don't have keys

parsed = json.loads(raw)        # now it is a dict
parsed["category"]              # 'URGENT' ✓
```

### LLMs sometimes wrap JSON in markdown fences

```
```json
{"category": "URGENT"}
```
```

Always strip the fences before calling `json.loads()` — the function below handles this.


### `parse_llm_json_response()` — string → dict, handles fences


In [ ]:
def parse_llm_json_response(raw_response: str) -> dict:
    """
    Parse a JSON string returned by an LLM into a Python dict.

    The raw LLM response is ALWAYS a string — even when the LLM was
    told to return JSON. You must call json.loads() to get a dict.

    Also handles markdown fences that LLMs sometimes add:
        ```json
        {"key": "value"}
        ```
    """
    # Step 1: remove leading/trailing whitespace from the entire response
    cleaned = raw_response.strip()

    # Step 2: detect and strip markdown code fences
    # LLMs instructed to return JSON often wrap it in ```json ... ```
    # json.loads() cannot parse the backticks — we must remove them first
    if cleaned.startswith("```"):
        # .split('\n', 1) splits on the FIRST newline only → ['```json', '{...}\n```']
        # [-1] takes everything AFTER the opening fence line
        cleaned = cleaned.split("\n", 1)[-1]

        # .rsplit('```', 1) splits on the LAST occurrence of ``` → ['{...}\n', '']
        # [0] takes everything BEFORE the closing fence
        cleaned = cleaned.rsplit("```", 1)[0]

        # .strip() again — removes any whitespace or newlines left after fence removal
        cleaned = cleaned.strip()

    # Step 3: parse the clean JSON string into a Python dict
    # json.loads()  →  string → Python object (dict, list, str, int, float, bool, None)
    # Raises json.JSONDecodeError if the string is not valid JSON
    # (e.g. LLM hallucinated text before the JSON, or forgot closing brace)
    parsed = json.loads(cleaned)

    return parsed


In [ ]:
# ── Demo: parse_llm_json_response() — plain JSON ───────────
raw_clean = '{"category": "TRACK_ORDER", "confidence": "high", "reason": "Customer asked where order is"}'

print(f"Before parse — type : {type(raw_clean).__name__}")
# raw_clean['category'] → TypeError: string indices must be integers

# json.loads() converts JSON string → Python dict
parsed = parse_llm_json_response(raw_clean)

print(f"After parse  — type : {type(parsed).__name__}")
print(f"category  : {parsed.get('category',  'UNKNOWN')}")
print(f"confidence: {parsed.get('confidence','low')}")
print(f"missing   : {parsed.get('missing_field', 'N/A')}   ← key absent, default returned")


In [ ]:
# ── Demo: parse_llm_json_response() — fenced JSON ──────────
# LLMs very commonly wrap JSON in markdown fences even when told not to
fenced = '```json\n{"intent": "TRACK_ORDER", "urgency": "high"}\n```'

print("Raw string from LLM:")
print(fenced)
print()

# TRACE through parse_llm_json_response(fenced):
# Step 1: cleaned = fenced.strip()
#         '```json\n{"intent": "TRACK_ORDER", "urgency": "high"}\n```'
# Step 2: startswith('```') → True
#         split('\n',1)[-1]   → '{"intent": "TRACK_ORDER", "urgency": "high"}\n```'
#         rsplit('```',1)[0]  → '{"intent": "TRACK_ORDER", "urgency": "high"}\n'
#         .strip()            → '{"intent": "TRACK_ORDER", "urgency": "high"}'
# Step 3: json.loads()       → {'intent': 'TRACK_ORDER', 'urgency': 'high'}

parsed2 = parse_llm_json_response(fenced)
print(f"After parse: {parsed2}")
print(f"intent    : {parsed2.get('intent')}")


### `safe_get_field()` — never raises KeyError on LLM output


In [ ]:
def safe_get_field(
    data: dict,
    field: str,
    default: Any = None,
    expected_type: type | None = None,
) -> Any:
    """
    Safely extract a field from a parsed LLM response dict.

    Never use data['field'] directly on LLM output:
        data['category']         → KeyError if LLM omitted the field
        data.get('category','?') → '?' safely — service stays running
    """
    # .get(field, default) is the safe alternative to data[field]
    # If 'field' is missing → returns default instead of raising KeyError
    # This is critical for LLM output — the LLM does not always comply
    value = data.get(field, default)

    # Optional type coercion — convert the value to expected_type if provided
    # Example: expected_type=int converts '1024' (str from JSON) → 1024 (int)
    # If the value is None (i.e. was missing and default=None) skip coercion
    if expected_type is not None and value is not None:
        try:
            value = expected_type(value)   # e.g. int('1024') → 1024
        except (ValueError, TypeError):
            # Coercion failed (e.g. int('high') → ValueError)
            # Fall back to the default rather than crashing the service
            value = default

    return value


In [ ]:
# ── Demo: safe_get_field() ──────────────────────────────────
llm_response = {"category": "TRACK_ORDER", "confidence": "high"}
# Note: 'reason' key is MISSING from this LLM response

# d['key']           → KeyError if key missing → CRASHES the service
# d.get('key', '?')  → returns '?' safely      → service keeps running
category   = safe_get_field(llm_response, 'category',   'UNKNOWN')
confidence = safe_get_field(llm_response, 'confidence', 'low')
reason     = safe_get_field(llm_response, 'reason',     'not provided')  # missing

print(f"category  : {category}")
print(f"confidence: {confidence}")
print(f"reason    : {reason}   ← key missing, default returned safely")

# json.dumps demo — dict → string (opposite direction)
model_config = {'model': 'gpt-4o', 'max_tokens': 1024, 'temperature': 0.2}
print()
print("json.dumps() compact :", json.dumps(model_config))
print("json.dumps() pretty  :")
print(json.dumps(model_config, indent=2))


---


## 4 — Reading CSV Data

`csv.DictReader` reads each row as a dict keyed by the column headers — the same shape as a database row in Python. This is exactly the format Day 08 returns from Postgres queries.

```python
with open('customers.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # row = {'customer_id': '1001', 'name': 'Prudhvi', ...}
        print(row['name'])
```

> Note: CSV values are always **strings**. Convert types explicitly: `int(row['customer_id'])`, `float(row['price'])`.


### `load_customers()` — each CSV row as a dict


In [ ]:
def load_customers(limit: int = 10) -> list[dict]:
    """
    Load customer records from customers.csv.
    Returns a list of dicts — one dict per row, keys = column headers.

    IMPORTANT: all CSV values are STRINGS.
    Convert types manually: int(row['customer_id']), float(row['price'])
    """
    filepath  = DATA_DIR / "customers.csv"
    customers = []   # accumulator — starts empty, grows with each row

    with open(filepath, "r", encoding="utf-8") as f:
        # csv.DictReader reads the FIRST row as column headers
        # then maps each subsequent row's values to those headers
        # Result: each row → {'customer_id': '1001', 'name': 'Prudhvi', ...}
        reader = csv.DictReader(f)

        # ITERATION TRACE:
        # Iter 1 → row = {'customer_id':'1001','name':'Prudhvi Akella','city':'Rajahmundry',...}
        #           limit=10 → len(customers)=0 < 10 → append → customers has 1 item
        # Iter 2 → row = {'customer_id':'1002','name':'Ravi Kumar','city':'Hyderabad',...}
        #           len(customers)=1 < 10 → append → customers has 2 items
        # ...
        # Iter 10 → len(customers)=9 < 10 → append → customers has 10 items
        # Iter 11 → limit=10, len(customers)=10 → break
        for row in reader:
            customers.append(dict(row))   # dict(row) copies the OrderedDict into a plain dict
            if limit and len(customers) >= limit:
                break   # stop early — avoid loading the entire file into memory

    return customers


In [ ]:
# ── Demo: load_customers() — csv.DictReader ────────────────
import io, csv

# io.StringIO wraps a string so csv.DictReader treats it like an open file
sample_csv = """customer_id,name,email,city,state
1001,Prudhvi Akella,prudhvi@example.com,Rajahmundry,AP
1002,Ravi Kumar,ravi@example.com,Hyderabad,TS
1003,Ananya Sharma,ananya@example.com,Bangalore,KA
"""

# csv.DictReader reads row 1 as headers, maps every subsequent row to those headers
# ITERATION TRACE:
# Row 1 → {'customer_id':'1001','name':'Prudhvi Akella','city':'Rajahmundry','state':'AP'}
# Row 2 → {'customer_id':'1002','name':'Ravi Kumar',   'city':'Hyderabad',  'state':'TS'}
# Row 3 → {'customer_id':'1003','name':'Ananya Sharma','city':'Bangalore',  'state':'KA'}
# EOF   → loop ends
reader    = csv.DictReader(io.StringIO(sample_csv))
customers = list(reader)

for c in customers:
    print(f"  {c['customer_id']}  {c['name']:20s}  {c['city']}, {c['state']}")

print()
# CSV trap — all values are strings
print(f"customer_id type     : {type(customers[0]['customer_id']).__name__}  ← str, not int")
print(f"String compare trap  : '1001' > '200' = {'1001' > '200'}  ← WRONG")
print(f"Correct int compare  : 1001  > 200  = {1001  > 200}")
print(f"Convert with int()   : {int(customers[0]['customer_id'])}")


### `load_few_shot_examples()` + `format_few_shot_examples()`


In [ ]:
def load_few_shot_examples(filename: str = "few_shot_examples.json") -> list[dict]:
    """
    Load few-shot examples from a JSON file in the prompts/ directory.

    File format expected:
    {
        "examples": [
            {"input": "Where is my order?", "output": "TRACK_ORDER"},
            ...
        ]
    }
    """
    filepath = PROMPT_DIR / filename

    with open(filepath, "r", encoding="utf-8") as f:
        # json.load(f) reads JSON directly from a file object
        # difference from json.loads():
        #   json.loads(string) → takes a string as input
        #   json.load(file)    → reads from an open file object (more memory efficient)
        data = json.load(f)

    # .get('examples', []) safely handles:
    #   - key exists         → returns the list of examples
    #   - key missing        → returns [] instead of KeyError
    #   - empty file/bad JSON→ caught upstream by the with block
    return data.get("examples", [])


In [ ]:
def format_few_shot_examples(examples: list[dict]) -> str:
    """
    Format few-shot examples into a string for the ## Examples section
    of the system prompt.
    """
    lines = []   # accumulate formatted lines — join at the end

    # ITERATION TRACE:
    # Iter 1 → ex = {'input': 'Where is my order?', 'output': 'TRACK_ORDER'}
    #           lines.append('Input : Where is my order?')
    #           lines.append('Output: TRACK_ORDER')
    #           lines.append('')               ← blank line separator
    #           lines = ['Input : Where...', 'Output: TRACK_ORDER', '']
    #
    # Iter 2 → ex = {'input': 'I want to cancel', 'output': 'CANCEL'}
    #           lines = [..., 'Input : I want to cancel', 'Output: CANCEL', '']
    for ex in examples:
        lines.append(f"Input : {ex['input']}")
        lines.append(f"Output: {ex['output']}")
        lines.append("")   # blank line visually separates each example pair

    # '\n'.join(lines) → 'Input : Where...\nOutput: TRACK_ORDER\n\nInput : I want...\n...'
    # This string is ready to paste into the ## Examples section of the system prompt
    return "\n".join(lines)


In [ ]:
# ── Demo: load_few_shot_examples() + format_few_shot_examples() ─
few_shot_data = {
    'examples': [
        {'input': 'Where is my order #3042?',   'output': 'TRACK_ORDER'},
        {'input': 'I want to cancel my purchase','output': 'CANCEL'},
        {'input': 'Do you have a discount code?','output': 'PROMOTIONS'},
    ]
}

# .get('examples', []) returns [] if key missing — never KeyError
examples = few_shot_data.get('examples', [])

# format_few_shot_examples() builds the ## Examples section string
# ITERATION TRACE:
# Iter 1 → ex={'input':'Where is my order #3042?','output':'TRACK_ORDER'}
#           lines → ['Input : Where is my order #3042?', 'Output: TRACK_ORDER', '']
# Iter 2 → ex={'input':'I want to cancel...','output':'CANCEL'}
#           lines → [..., 'Input : I want to cancel...', 'Output: CANCEL', '']
# Iter 3 → ex={'input':'Do you have...','output':'PROMOTIONS'}
#           lines → [..., 'Input : Do you have...', 'Output: PROMOTIONS', '']
lines = []
for ex in examples:
    lines.append(f"Input : {ex['input']}")
    lines.append(f"Output: {ex['output']}")
    lines.append('')   # blank line separator between pairs

formatted = '\n'.join(lines)
print("## Examples section (ready to paste into system prompt):")
print(formatted)


---


## 5 — String Methods for Prompt Engineering

| Method | What it does | Example |
|--------|-------------|---------|
| `.strip()` | Remove leading/trailing whitespace and newlines | `'  hello  '.strip()` → `'hello'` |
| `.split()` | Split on any whitespace, removes empty strings | `'a  b'.split()` → `['a', 'b']` |
| `' '.join(list)` | Join list back into string with separator | `' '.join(['a','b'])` → `'a b'` |
| `.lower()` | Convert to lowercase | `'CANCEL'.lower()` → `'cancel'` |
| `.replace(a, b)` | Replace all occurrences of a with b | `'hi\nbye'.replace('\n',' ')` |
| `.startswith(s)` | Does string start with s? | `'```json'.startswith('```')` → `True` |
| `len(s)` | Number of characters | `len('hello')` → `5` |
| `s[:n]` | First n characters (truncate) | `s[:300]` → first 300 chars |

### The clean-then-inject pattern

Raw text from a database often has extra whitespace, tabs, and newlines.  
Always clean before injecting into a prompt — noise wastes tokens and confuses the LLM.

```python
text = raw.strip()          # 1. remove leading/trailing whitespace
text = ' '.join(text.split())  # 2. collapse internal whitespace to single spaces
text = text[:300]           # 3. truncate to token budget
```


### `clean_product_description()` — strip · collapse · truncate


In [ ]:
def clean_product_description(raw_description: str, max_length: int = 300) -> str:
    """
    Clean and truncate a product description for injection into an LLM prompt.

    Raw database text typically has extra spaces, tabs, and newlines.
    Cleaning prevents wasted tokens and confusing the LLM with formatting noise.
    """
    # Step 1: .strip() — remove leading and trailing whitespace/newlines
    # '  Hello world  \n' → 'Hello world'
    text = raw_description.strip()

    # Step 2: Collapse all internal whitespace to single spaces
    # .split() with NO argument splits on ANY whitespace (space, \t, \n, \r)
    #   and automatically removes empty strings from multiple consecutive spaces
    # Example: 'Hello   World\n\nFoo' → ['Hello', 'World', 'Foo']
    # ' '.join() reassembles with exactly one space between each word
    # Result:   'Hello World Foo'
    words = text.split()       # ['Hello', 'World', 'Foo']
    text  = " ".join(words)   # 'Hello World Foo'

    # Step 3: Truncate to the token budget
    # max_length - 3 reserves space for the '...' suffix
    # len(text) > max_length check prevents adding '...' to a short string
    if len(text) > max_length:
        text = text[:max_length - 3] + "..."   # 'Hello World...' signals truncation to LLM

    return text


In [ ]:
# ── Demo: clean_product_description() ──────────────────────
raw = '  Classic Monitor  —   Beautiful   27-inch   4K   display.\n\n'\
      '  Perfect   for   home   office   use.  '

print(f"Original ({len(raw)} chars):")
print(repr(raw))   # repr() shows \n and spaces explicitly
print()

# Step 1: .strip() removes leading/trailing whitespace/newlines
step1 = raw.strip()
print(f"Step 1 — .strip() ({len(step1)} chars):")
print(repr(step1))
print()

# Step 2: .split() with no args splits on ANY whitespace and drops empty strings
# TRACE: ['Classic','Monitor','—','Beautiful','27-inch','4K','display.','Perfect','for','home','office','use.']
# ' '.join() reassembles with exactly one space between each word
step2 = ' '.join(step1.split())
print(f"Step 2 — collapse whitespace ({len(step2)} chars):")
print(repr(step2))
print()

# Step 3: truncate — max_length-3 reserves space for '...'
MAX = 50
step3 = step2[:MAX-3] + '...' if len(step2) > MAX else step2
print(f"Step 3 — truncate to {MAX} chars:")
print(repr(step3))


### `build_product_context_for_prompt()` — format context block for prompt


In [ ]:
def build_product_context_for_prompt(product: dict) -> str:
    """
    Format a product dict into a <context> block for the LLM prompt.

    XML tags (<context>) tell the LLM this is retrieved data, not user input.
    This follows the Module 00 RAG prompt anatomy from Technique 05.
    """
    # .get(key, default) on every field — defensive against missing keys
    # If the database row is missing 'price', .get('price', 0) returns 0
    # instead of crashing with KeyError
    #
    # :.2f inside the f-string formats price as exactly 2 decimal places
    # product.get('price', 0):.2f  →  205.21  (not 205.209999...)
    context = f"""<context>
Product ID   : {product.get('product_id',    'N/A')}
Product Name : {product.get('product_name',  'Unknown')}
Category     : {product.get('category',      'Unknown')}
Brand        : {product.get('brand',         'Unknown')}
Price        : ${product.get('price',         0):.2f}
In Stock     : {product.get('stock_quantity', 0)} units
Description  : {clean_product_description(product.get('description', ''))}
</context>"""

    # Note: clean_product_description() is called inline inside the f-string
    # Python evaluates it, then substitutes the returned string into the template
    return context


In [ ]:
# ── Demo: build_product_context_for_prompt() ───────────────
product = {
    'product_id'    : 'PROD-101',
    'product_name'  : 'Classic Monitor',
    'category'      : 'Electronics',
    'brand'         : 'ViewPro',
    'price'         : 205.21,
    'stock_quantity': 238,
    'description'   : '  Beautiful   27-inch   4K   display   for   home   office.  ',
    # 'rating' key intentionally missing — .get() handles safely
}

# .get(key, default) on every field — no crash if database column is absent
# TRACE of .get() calls:
# product.get('product_id', 'N/A')        → 'PROD-101'  (key present)
# product.get('rating',     'N/A')        → 'N/A'       (key MISSING — safe)
# product.get('price', 0):.2f             → '205.21'    (formatted float)
# clean_product_description(description)  → 'Beautiful 27-inch 4K display...'

context = f"""<context>
Product ID   : {product.get('product_id',    'N/A')}
Product Name : {product.get('product_name',  'Unknown')}
Category     : {product.get('category',      'Unknown')}
Brand        : {product.get('brand',         'Unknown')}
Price        : ${product.get('price',         0):.2f}
In Stock     : {product.get('stock_quantity', 0)} units
Description  : {' '.join(product.get('description','').strip().split())}
</context>"""

print(context)
print()
print("The <context> block is injected into the system prompt.")
print("The LLM uses it to answer product questions without inventing details.")


---


## 6 — Full Pipeline Demo

All four skills combined: load a prompt from file → read customer from CSV → parse LLM JSON response → build the next request.


In [ ]:
import io

# ── Step 1: Load system prompt from .txt file ──────────────
# In production: open(PROMPT_DIR / 'system_prompt.txt') reads from disk
# Here: we inline the string to run without the file system
system_prompt = """
## Role
You are a customer support assistant for ShopSmart e-commerce.

## Task
Answer using ONLY the information provided. Never invent details.

## Output Format
Return JSON: {"category": str, "reply": str, "confidence": str}
""".strip()   # .strip() removes leading/trailing newlines — same as load_system_prompt()

# ── Step 2: Load customer row from CSV ─────────────────────
# In production: csv.DictReader opens customers.csv
# Here: one row dict to simulate the result
csv_row       = {'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
customer_id   = int(csv_row['customer_id'])   # CSV gives str → convert to int
customer_name = csv_row['name']               # str — no conversion needed

# ── Step 3: Build user message using f-string (Day 01) ─────
# This is exactly the pattern from Day 01 Section 4
user_message  = f"Customer: {customer_name}\nQuery: Where is my order #3042?"

# ── Step 4: Simulate LLM JSON response with markdown fences ─
# In production: this comes back from openai.chat.completions.create()
mock_llm_output = '```json\n{"category": "TRACK_ORDER", "reply": "Your order ORD-3042 is In Transit, arriving Friday.", "confidence": "high"}\n```'

# ── Step 5: Strip fences + parse JSON ──────────────────────
# TRACE:
# startswith('```') → True
# split('\n',1)[-1] → '{"category":...}\n```'
# rsplit('```',1)[0] → '{"category":...}\n'
# .strip()           → '{"category": "TRACK_ORDER", "reply": "...", "confidence": "high"}'
# json.loads()       → Python dict
cleaned = mock_llm_output.strip()
if cleaned.startswith('```'):
    cleaned = cleaned.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
parsed_response = json.loads(cleaned)

# ── Step 6: Extract fields safely with .get() ───────────────
# .get() never raises KeyError — critical for production LLM services
category   = parsed_response.get('category',   'UNKNOWN')
reply      = parsed_response.get('reply',       'No reply provided')
confidence = parsed_response.get('confidence',  'unknown')

# ── Step 7: Log the call as JSONL ───────────────────────────
# json.dumps converts dict → string for writing to the log file
log_entry = json.dumps({
    'customer_id': customer_id,
    'customer'   : customer_name,
    'category'   : category,
    'confidence' : confidence,
})

# ── Output ──────────────────────────────────────────────────
print(f"System prompt : {len(system_prompt)} chars loaded")
print(f"Customer      : {customer_name} (id={customer_id}, type={type(customer_id).__name__})")
print(f"User message  : {user_message}")
print(f"LLM category  : {category}")
print(f"LLM reply     : {reply}")
print(f"Confidence    : {confidence}")
print(f"Log entry     : {log_entry}")


---


## Summary — Day 04

| Concept | Key rule / syntax |
|---------|-------------------|
| `with open(...) as f` | Safe file handling — always closes even on error |
| `__file__` in Jupyter | **NameError** — use `try/except` and fall back to `Path.cwd()` |
| Mode `"r"` | Read existing file (default) |
| Mode `"w"` | Write — overwrites if file exists |
| Mode `"a"` | Append — creates file if missing |
| `encoding="utf-8"` | Always specify — avoids platform differences |
| `json.loads(str)` | JSON string → Python dict (parse LLM response) |
| `json.dumps(dict)` | Python dict → JSON string (log entries, API payloads) |
| `json.load(file)` | Read JSON directly from file object |
| Strip markdown fences | `if cleaned.startswith('```')` before `json.loads()` |
| `.get(key, default)` | Safe dict access — never raises `KeyError` on LLM output |
| `csv.DictReader` | Each CSV row → dict with column headers as keys |
| CSV values are strings | Always convert: `int(row['id'])`, `float(row['price'])` |
| `.strip()` | Remove leading/trailing whitespace |
| `' '.join(s.split())` | Collapse all internal whitespace to single spaces |
| `s[:n] + '...'` | Truncate text to token budget |
| JSONL format | One JSON object per line — machine-readable log files |

**Connection to previous days:**
- f-strings (Day 01) fill the prompt templates loaded from `.txt` files
- Dicts and `.get()` (Day 01/03) handle the parsed LLM JSON response safely
- Functions (Day 03) wrap every file operation for reuse across the codebase

**Next:** Day 05 — Async Python  
`async def` · `await` · `asyncio.gather()` · calling the OpenAI API asynchronously
